# 7R-C. Evaluate one Environment B world

This notebook evaluates one feature package produced by 7R-B, using the primary detector settings transferred from Environment A. It does not select a new detector family on the retail validation results.

**Saved run:** `FINAL`, condition `B1_structured`, seed 2105. This is one reserved-final replication, not the primary seed-2101 result. The setup reads this status from the input package and prints it before evaluation.

Each configuration has its own training-fitted `RobustScaler` and Isolation Forest, with 300 trees, maximum sample size 2,048 and model seed 42. The primary budget selects the top 1.0% of user-days in the completed test window.

**Criterion A** compares false-positive rates at M0-B's recall anchor. A favourable M2-B-minus-M0-B result requires the paired 95% interval to lie below zero. **Criterion B** compares recall at equal alert capacity and requires the interval to lie above zero. A favourable point estimate alone is insufficient. M3-B is checked separately for a supported favourable improvement.

The code uses `M0`–`M3` as within-notebook identifiers; the dissertation adds `-B` to distinguish this environment.

**Outputs:** fitted models, scores, evaluation tables, paired intervals, diagnostics, a result card and integrity records for this world. The overall H1-B decision follows the separately documented seed-2101 rule, not whichever world is loaded here.

*Documentation copy, 8 November 2026. Notes edited for clarity; code, saved outputs and execution metadata are unchanged. This copy has not been rerun.*


In [1]:

# ============================================================
# 0. Setup + discover one frozen 7R-B package
# ============================================================
import os, glob, gc, json, time, random, hashlib, shutil, warnings, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score
import joblib

SEED = 42
K_PRIMARY = 1.0
K_GRID = [0.05, 0.1, 0.25, 0.5, 1.0, 2.0]
N_BOOT_BUDGET = 1000
N_BOOT_MATCHED = 300
IF_ESTIMATORS = 300
IF_MAX_SAMPLES = 2048
SCORE_CHUNK = 200_000

random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings('ignore')

WORK=Path('/kaggle/working')
T0=time.time()

hits=sorted(glob.glob('/kaggle/input/**/features_cyberlog_retail.parquet',recursive=True))
assert hits, (
    'Attach the successful 7R-B Kaggle output/dataset before running 7R-C. '
    'Expected features_cyberlog_retail.parquet.'
)

required=[
 'features_cyberlog_retail.parquet',
 'context_curated_retail.parquet',
 'context_rawcat_retail.parquet',
 'context_shuffled_retail.parquet',
 'labels_retail.parquet',
 'retail_7rb_feature_contract.json',
 'retail_7rb_summary.json',
 'retail_7rb_output_sha256.csv',
]
ROOT=None
for h in hits:
    d=Path(h).parent
    if all((d/f).exists() for f in required):
        ROOT=d; break
assert ROOT is not None,'No complete 7R-B input directory found.'

summary=json.load(open(ROOT/'retail_7rb_summary.json'))
contract=json.load(open(ROOT/'retail_7rb_feature_contract.json'))
SOURCE_MODE=str(summary['source_run_mode']).upper()
SOURCE_CONDITION=str(summary['source_condition'])
SOURCE_SEED=int(summary['source_seed'])

OUT=WORK/f"retail_7rc_{SOURCE_MODE.lower()}_{SOURCE_CONDITION}_seed{SOURCE_SEED}"
QA=OUT/'qa'; MODELS=OUT/'models'; FIGS=OUT/'figures'
for p in (OUT,QA,MODELS,FIGS): p.mkdir(parents=True,exist_ok=True)

print('7R-B input root :',ROOT)
print('7R-C output     :',OUT)
print('Source mode     :',SOURCE_MODE)
print('Source condition:',SOURCE_CONDITION)
print('Source seed     :',SOURCE_SEED)
print('Python          :',platform.python_version())
print('scikit-learn    :',sklearn.__version__)
if SOURCE_MODE=='DEV':
    print('\nDEVELOPMENT RUN: any positive criterion outcome is diagnostic only, not final H1-B evidence.')
else:
    print('\nRESERVED-FINAL RUN: this remains one per-run result; overall H1-B requires the frozen final-world rule.')


7R-B input root : /kaggle/input/notebooks/thandondhlovu10/7r-b-retail-environment-b-feature-context-pipeli/retail_7rb
7R-C output     : /kaggle/working/retail_7rc_final_B1_structured_seed2105
Source mode     : FINAL
Source condition: B1_structured
Source seed     : 2105
Python          : 3.12.13
scikit-learn    : 1.6.1

RESERVED-FINAL RUN: this remains one per-run result; overall H1-B requires the frozen final-world rule.


In [2]:

# ============================================================
# 1. Verify every 7R-B recorded SHA-256 hash before use
# ============================================================
def sha256_file(path,block=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as fh:
        while True:
            b=fh.read(block)
            if not b: break
            h.update(b)
    return h.hexdigest()

mf=pd.read_csv(ROOT/'retail_7rb_output_sha256.csv')
verify=[]
for r in mf.itertuples(index=False):
    p=ROOT/r.relative_path
    actual=sha256_file(p) if p.exists() else None
    verify.append({'relative_path':r.relative_path,'exists':p.exists(),'expected':r.sha256,
                   'actual':actual,'passed':bool(p.exists() and actual==r.sha256)})
verify_df=pd.DataFrame(verify)
verify_df.to_csv(QA/'7rb_input_hash_verification.csv',index=False)
print(f"7R-B hashes: {int(verify_df.passed.sum())}/{len(verify_df)} passed")
assert verify_df.passed.all(),'7R-B input hash verification failed. STOP.'


7R-B hashes: 14/14 passed


In [3]:

# ============================================================
# 2. Load frozen feature blocks + labels
# ============================================================
tech=pd.read_parquet(ROOT/'features_cyberlog_retail.parquet')
raw=pd.read_parquet(ROOT/'context_rawcat_retail.parquet')
cur=pd.read_parquet(ROOT/'context_curated_retail.parquet')
shuf=pd.read_parquet(ROOT/'context_shuffled_retail.parquet')
labels=pd.read_parquet(ROOT/'labels_retail.parquet')

for df in (tech,raw,cur,shuf,labels):
    df['user']=df['user'].astype(str)
    df['day']=pd.to_datetime(df['day']).dt.normalize()

KEY=['user','day']
assert not tech.duplicated(KEY).any()
assert len(tech)==len(raw)==len(cur)==len(shuf)==len(labels)

# Labels are merged for evaluation only, after all feature blocks are already frozen.
base=tech.merge(labels[KEY+['label']],on=KEY,how='left',validate='one_to_one')
base['label']=base['label'].fillna(0).astype('int8')
assert base['partition'].equals(tech['partition'])

M0_FEATURES=contract['m0_feature_names']
M1_CTX=[c for c in raw.columns if c not in KEY]
M2_CTX=[c for c in cur.columns if c not in KEY]
M3_CTX=[c for c in shuf.columns if c not in KEY]

assert len(M0_FEATURES)==41
assert len(M2_CTX)==18 and len(M3_CTX)==18
assert len(M1_CTX)==int(contract['m1_context_feature_count'])

# Align every context block strictly by the same keys.
def align_ctx(df,cols):
    x=base[KEY].merge(df[KEY+cols],on=KEY,how='left',validate='one_to_one')
    assert x[cols].notna().all().all()
    return x[cols]

X0=base[M0_FEATURES].astype('float32')
X1=pd.concat([X0.reset_index(drop=True),align_ctx(raw,M1_CTX).reset_index(drop=True)],axis=1)
X2=pd.concat([X0.reset_index(drop=True),align_ctx(cur,M2_CTX).reset_index(drop=True)],axis=1)
X3=pd.concat([X0.reset_index(drop=True),align_ctx(shuf,M3_CTX).reset_index(drop=True)],axis=1)

FEATURES={'M0':list(X0.columns),'M1':list(X1.columns),'M2':list(X2.columns),'M3':list(X3.columns)}
X={'M0':X0,'M1':X1,'M2':X2,'M3':X3}

tr=base.partition.eq('train').to_numpy()
va=base.partition.eq('validation').to_numpy()
te=base.partition.eq('test').to_numpy()
y=base.label.to_numpy('int8')
users=base.user.to_numpy()

print('rows:',len(base),'train:',tr.sum(),'validation:',va.sum(),'test:',te.sum())
print('positive days:',{p:int(base.loc[base.partition.eq(p),'label'].sum()) for p in ['train','validation','test']})
print('test malicious accounts:',base.loc[te & base.label.eq(1),'user'].nunique())
print('feature dimensions:',{k:len(v) for k,v in FEATURES.items()})
assert int(y[tr].sum())==0,'Training is not benign-only. STOP.'


rows: 264893 train: 132574 validation: 44957 test: 87362
positive days: {'train': 0, 'validation': 90, 'test': 180}
test malicious accounts: 60
feature dimensions: {'M0': 41, 'M1': 61, 'M2': 59, 'M3': 59}


## 3. Fit the transferred primary detector

After checking the input hashes and loading aligned feature tables, fit a separate scaler and Isolation Forest to each configuration's training matrix. The detector family, settings and seed are shared; the fitted objects are not.

No outcome labels, validation metrics, test metrics or scenario identities are used to select a family or tune its settings here. The following cells score the configurations and report the primary fixed-budget results and descriptive budget grid.


In [4]:

# ============================================================
# 3. Fit M0/M1/M2/M3 and save anomaly scores
# ============================================================
def score_in_chunks(model,arr,chunk=SCORE_CHUNK):
    out=np.empty(len(arr),dtype='float64')
    for a in range(0,len(arr),chunk):
        b=min(a+chunk,len(arr))
        # sklearn IsolationForest: lower score_samples = more anomalous.
        out[a:b]=-model.score_samples(arr[a:b])
    return out

S={}
fit_records=[]
for cfg in ('M0','M1','M2','M3'):
    print('\nFitting',cfg,'with',X[cfg].shape[1],'features')
    scaler=RobustScaler()
    xtr=scaler.fit_transform(X[cfg].loc[tr]).astype('float32')
    model=IsolationForest(
        n_estimators=IF_ESTIMATORS,
        max_samples=IF_MAX_SAMPLES,
        contamination='auto',
        max_features=1.0,
        bootstrap=False,
        n_jobs=-1,
        random_state=SEED,
    )
    model.fit(xtr)
    joblib.dump(scaler,MODELS/f'{cfg}_robust_scaler.joblib')
    joblib.dump(model,MODELS/f'{cfg}_isolation_forest.joblib')

    all_scores=np.empty(len(base),dtype='float64')
    for mask_name,mask in [('train',tr),('validation',va),('test',te)]:
        idx=np.flatnonzero(mask)
        vals=scaler.transform(X[cfg].iloc[idx]).astype('float32')
        all_scores[idx]=score_in_chunks(model,vals)
    S[cfg]=all_scores
    fit_records.append({'config':cfg,'features':X[cfg].shape[1],
                        'train_rows':int(tr.sum()),'n_estimators':IF_ESTIMATORS,
                        'max_samples':IF_MAX_SAMPLES,'seed':SEED})
    del xtr,model,scaler
    gc.collect()

score_frame=base[KEY+['partition','label']].copy()
for cfg in S: score_frame[f'{cfg}_if']=S[cfg]
score_frame.to_parquet(OUT/'scores_retail_all_models.parquet',index=False)
pd.DataFrame(fit_records).to_csv(OUT/'model_fit_record.csv',index=False)
print('all scores finite:',all(np.isfinite(S[c]).all() for c in S))



Fitting M0 with 41 features

Fitting M1 with 61 features

Fitting M2 with 59 features

Fitting M3 with 59 features
all scores finite: True


In [5]:

# ============================================================
# 4. Primary fixed-budget results + ranking metrics
# ============================================================
def top_budget_alert(score,k_pct):
    n=max(1,int(np.floor(len(score)*k_pct/100.0)))
    order=np.argsort(-np.asarray(score),kind='mergesort')
    alert=np.zeros(len(score),dtype=bool)
    alert[order[:n]]=True
    return alert

def counts_metrics(y,alert):
    y=np.asarray(y).astype(bool); a=np.asarray(alert).astype(bool)
    tp=int((y&a).sum()); fp=int((~y&a).sum()); fn=int((y&~a).sum()); tn=int((~y&~a).sum())
    rec=tp/(tp+fn) if tp+fn else np.nan
    prec=tp/(tp+fp) if tp+fp else np.nan
    fpr=fp/(fp+tn) if fp+tn else np.nan
    return {'tp':tp,'fp':fp,'fn':fn,'tn':tn,'recall':rec,'precision':prec,'fpr':fpr}

yt=y[te]; ut=users[te]
St={c:S[c][te] for c in S}
ALERT={c:top_budget_alert(St[c],K_PRIMARY) for c in S}

rows=[]
for cfg in ('M0','M1','M2','M3'):
    m=counts_metrics(yt,ALERT[cfg])
    m.update({
        'config':cfg,'budget_pct':K_PRIMARY,'alerts':int(ALERT[cfg].sum()),
        'roc_auc':float(roc_auc_score(yt,St[cfg])),
        'average_precision':float(average_precision_score(yt,St[cfg])),
    })
    rows.append(m)
primary=pd.DataFrame(rows)[['config','budget_pct','alerts','tp','fp','fn','tn','recall','precision','fpr','roc_auc','average_precision']]
primary.to_csv(OUT/'primary_results_retail.csv',index=False)
display(primary)

alert_counts={c:int(ALERT[c].sum()) for c in ALERT}
assert len(set(alert_counts.values()))==1,alert_counts


,config,budget_pct,alerts,tp,fp,fn,tn,recall,precision,fpr,roc_auc,average_precision
0,M0,1.0,873,45,828,135,86354,0.250000,0.051546,0.009497,0.829829,0.127906
1,M1,1.0,873,48,825,132,86357,0.266667,0.054983,0.009463,0.863082,0.170382
2,M2,1.0,873,47,826,133,86356,0.261111,0.053837,0.009474,0.851305,0.183635
3,M3,1.0,873,42,831,138,86351,0.233333,0.048110,0.009532,0.798654,0.138575


In [6]:

# ============================================================
# 5. Descriptive alert-budget grid
# ============================================================
curve_rows=[]
for k in K_GRID:
    for cfg in ('M0','M1','M2','M3'):
        a=top_budget_alert(St[cfg],k)
        m=counts_metrics(yt,a)
        curve_rows.append({'budget_pct':k,'config':cfg,'alerts':int(a.sum()),**m})
curves=pd.DataFrame(curve_rows)
curves.to_csv(OUT/'budget_curves_retail.csv',index=False)
display(curves.pivot(index='budget_pct',columns='config',values='recall'))


config,M0,M1,M2,M3
budget_pct,,,,
0.05,0.133333,0.161111,0.172222,0.150000
0.10,0.177778,0.194444,0.194444,0.183333
0.25,0.211111,0.216667,0.211111,0.205556
0.50,0.222222,0.244444,0.244444,0.216667
1.00,0.250000,0.266667,0.261111,0.233333
2.00,0.277778,0.316667,0.311111,0.255556


## 6. Distinguish malicious-day detection from account alert coverage

A malicious account counts as detected only when at least one of its labelled-malicious days is alerted.

The notebook also records whether a positive account receives any alert, including on a benign day. That broader coverage measure is reported separately and must not be substituted for malicious-activity detection.

The next cell compares M0-B and M2-B queue membership at the primary budget.


In [7]:

# ============================================================
# 6. Account coverage — corrected definition
# ============================================================
def account_metrics(users,y,alert):
    d=pd.DataFrame({'user':users,'y':np.asarray(y).astype(bool),'a':np.asarray(alert).astype(bool)})
    g=d.groupby('user').agg(
        positive_account=('y','any'),
        any_day_alerted=('a','any')
    )
    detected=(d['y'] & d['a']).groupby(d['user']).any()
    g['malicious_activity_detected']=detected.reindex(g.index).fillna(False)
    pos=g[g.positive_account]; ben=g[~g.positive_account]
    return {
        'malicious_accounts':int(len(pos)),
        'malicious_accounts_detected':int(pos.malicious_activity_detected.sum()),
        'account_malicious_activity_recall':float(pos.malicious_activity_detected.mean()) if len(pos) else np.nan,
        'positive_accounts_any_day_alerted':int(pos.any_day_alerted.sum()),
        'benign_accounts':int(len(ben)),
        'benign_accounts_alerted':int(ben.any_day_alerted.sum()),
        'benign_account_alert_rate':float(ben.any_day_alerted.mean()) if len(ben) else np.nan,
    }

acct=pd.DataFrame([{'config':cfg,**account_metrics(ut,yt,ALERT[cfg])} for cfg in ('M0','M1','M2','M3')])
acct.to_csv(OUT/'account_coverage_retail.csv',index=False)
display(acct)


,config,malicious_accounts,malicious_accounts_detected,account_malicious_activity_recall,positive_accounts_any_day_alerted,benign_accounts,benign_accounts_alerted,benign_account_alert_rate
0,M0,60,21,0.350000,31,1940,448,0.230928
1,M1,60,22,0.366667,32,1940,450,0.231959
2,M2,60,21,0.350000,30,1940,450,0.231959
3,M3,60,18,0.300000,27,1940,450,0.231959


In [8]:

# ============================================================
# 7. Queue overlap M0 vs M2
# ============================================================
a0=ALERT['M0']; a2=ALERT['M2']; yy=yt.astype(bool)
overlap={
 'shared_alerts':int((a0&a2).sum()),
 'm0_only_alerts':int((a0&~a2).sum()),
 'm2_only_alerts':int((a2&~a0).sum()),
 'shared_positive_alerts':int((a0&a2&yy).sum()),
 'm0_only_positive_alerts':int((a0&~a2&yy).sum()),
 'm2_only_positive_alerts':int((a2&~a0&yy).sum()),
 'shared_benign_alerts':int((a0&a2&~yy).sum()),
 'm0_only_benign_alerts':int((a0&~a2&~yy).sum()),
 'm2_only_benign_alerts':int((a2&~a0&~yy).sum()),
}
json.dump(overlap,open(OUT/'alert_overlap_m0_m2_retail.json','w'),indent=2)
print(json.dumps(overlap,indent=2))


{
  "shared_alerts": 702,
  "m0_only_alerts": 171,
  "m2_only_alerts": 171,
  "shared_positive_alerts": 44,
  "m0_only_positive_alerts": 1,
  "m2_only_positive_alerts": 3,
  "shared_benign_alerts": 658,
  "m0_only_benign_alerts": 170,
  "m2_only_benign_alerts": 168
}


## 8. Estimate paired uncertainty and apply the run-level rules

Users form the resampling clusters. Accounts with at least one malicious test day and accounts with none are sampled in separate strata, with the same sampled users used for both configurations.

For the fixed-budget comparison, retain the original alert decisions and resample the per-user counts for 1,000 replicates. For matched recall, use 300 replicates and find the ranked operating point again within each resample, retaining the observed M0-B recall anchor.

The subsequent verdict cell applies Criteria A and B to M2-B and checks M3-B for a favourable warning result. It saves a per-run outcome. The overall H1-B field remains pending for final runs because the separate primary-world rule must still be applied.


In [9]:

# ============================================================
# 8a. Fixed-budget paired bootstrap
# ============================================================
def per_user_counts(users,y,alert):
    d=pd.DataFrame({'user':users,'y':np.asarray(y).astype(bool),'a':np.asarray(alert).astype(bool)})
    d['tp']=(d.y&d.a).astype(int); d['fp']=(~d.y&d.a).astype(int)
    d['fn']=(d.y&~d.a).astype(int); d['tn']=(~d.y&~d.a).astype(int)
    return d.groupby('user')[['tp','fp','fn','tn']].sum()

def metrics_from_counts(tp,fp,fn,tn):
    return {
        'recall':tp/(tp+fn) if tp+fn else np.nan,
        'precision':tp/(tp+fp) if tp+fp else np.nan,
        'fpr':fp/(fp+tn) if fp+tn else np.nan,
    }

def paired_budget_delta(users,y,alert_ref,alert_new,n_boot=N_BOOT_BUDGET,seed=SEED):
    A=per_user_counts(users,y,alert_ref)
    B=per_user_counts(users,y,alert_new).reindex(A.index).fillna(0)
    pos_user=((A.tp+A.fn)>0)
    Ap,An=A[pos_user].to_numpy('int64'),A[~pos_user].to_numpy('int64')
    Bp,Bn=B[pos_user].to_numpy('int64'),B[~pos_user].to_numpy('int64')
    ref=metrics_from_counts(*[int(v) for v in A.sum()])
    new=metrics_from_counts(*[int(v) for v in B.sum()])
    rng=np.random.default_rng(seed)
    diffs={m:[] for m in ('recall','precision','fpr')}
    for _ in range(n_boot):
        ip=rng.integers(0,len(Ap),len(Ap)) if len(Ap) else np.array([],dtype=int)
        ib=rng.integers(0,len(An),len(An)) if len(An) else np.array([],dtype=int)
        ca=(Ap[ip].sum(axis=0) if len(ip) else np.zeros(4,dtype=int)) + (An[ib].sum(axis=0) if len(ib) else np.zeros(4,dtype=int))
        cb=(Bp[ip].sum(axis=0) if len(ip) else np.zeros(4,dtype=int)) + (Bn[ib].sum(axis=0) if len(ib) else np.zeros(4,dtype=int))
        ma=metrics_from_counts(*[int(v) for v in ca]); mb=metrics_from_counts(*[int(v) for v in cb])
        for m in diffs:
            diffs[m].append(mb[m]-ma[m])
    out={}
    for m,v in diffs.items():
        lo,hi=np.nanpercentile(v,[2.5,97.5])
        out[m]={'point':float(new[m]-ref[m]),'ci_lo':float(lo),'ci_hi':float(hi),'n_boot':n_boot}
    return out


In [10]:

# ============================================================
# 8b. Matched-recall FPR + paired bootstrap
# ============================================================
def fpr_at_recall_prefix(y,score,anchor):
    y=np.asarray(y).astype(bool); score=np.asarray(score,float)
    P=int(y.sum()); N=int((~y).sum())
    if P==0 or N==0: return np.nan,np.nan,0,0,0
    target=max(1,int(np.ceil(anchor*P-1e-12)))
    order=np.argsort(-score,kind='mergesort')
    cum=np.cumsum(y[order].astype(int))
    where=np.flatnonzero(cum>=target)
    if not len(where): return np.nan,np.nan,0,0,0
    stop=int(where[0])+1
    sel=order[:stop]
    tp=int(y[sel].sum()); fp=int((~y[sel]).sum())
    return fp/N,tp/P,stop,tp,fp

def paired_matched_fpr_delta(users,y,s_ref,s_new,anchor,n_boot=N_BOOT_MATCHED,seed=SEED):
    d=pd.DataFrame({'user':users,'y':np.asarray(y).astype(int),'ref':s_ref,'new':s_new})
    groups={u:(g.y.to_numpy(),g.ref.to_numpy(),g.new.to_numpy()) for u,g in d.groupby('user')}
    upos=[u for u,(yy,_,_) in groups.items() if yy.any()]
    uben=[u for u,(yy,_,_) in groups.items() if not yy.any()]
    assert upos and uben
    f0,r0,n0,tp0,fp0=fpr_at_recall_prefix(d.y,d.ref,anchor)
    f1,r1,n1,tp1,fp1=fpr_at_recall_prefix(d.y,d.new,anchor)
    rng=np.random.default_rng(seed)
    delta=[]
    for _ in range(n_boot):
        chosen=list(rng.choice(upos,len(upos),replace=True))+list(rng.choice(uben,len(uben),replace=True))
        yy=np.concatenate([groups[u][0] for u in chosen])
        sr=np.concatenate([groups[u][1] for u in chosen])
        sn=np.concatenate([groups[u][2] for u in chosen])
        fr,*_=fpr_at_recall_prefix(yy,sr,anchor)
        fn,*_=fpr_at_recall_prefix(yy,sn,anchor)
        if np.isfinite(fr) and np.isfinite(fn): delta.append(fn-fr)
    lo,hi=np.percentile(delta,[2.5,97.5])
    return {
        'point':float(f1-f0),'ci_lo':float(lo),'ci_hi':float(hi),
        'fpr_ref':float(f0),'fpr_new':float(f1),
        'achieved_recall_ref':float(r0),'achieved_recall_new':float(r1),
        'alerts_ref':int(n0),'alerts_new':int(n1),'tp_ref':int(tp0),'tp_new':int(tp1),
        'fp_ref':int(fp0),'fp_new':int(fp1),
        'n_boot_requested':int(n_boot),'n_boot_used':int(len(delta))
    }

ANCHOR=float(primary.loc[primary.config.eq('M0'),'recall'].iloc[0])
paired={}
for cfg in ('M1','M2','M3'):
    paired[cfg]={
        'fixed_budget':paired_budget_delta(ut,yt,ALERT['M0'],ALERT[cfg],seed=SEED),
        'matched_recall':paired_matched_fpr_delta(ut,yt,St['M0'],St[cfg],ANCHOR,seed=SEED),
    }
    b=paired[cfg]['fixed_budget']['recall']; a=paired[cfg]['matched_recall']
    print(f"\n{cfg} vs M0")
    print(f"  dRecall @1% : {b['point']:+.6f}  CI95 [{b['ci_lo']:+.6f}, {b['ci_hi']:+.6f}]")
    print(f"  dFPR matched: {a['point']:+.6f}  CI95 [{a['ci_lo']:+.6f}, {a['ci_hi']:+.6f}]")

json.dump({'anchor_recall':ANCHOR,'comparisons':paired},open(OUT/'paired_deltas_retail.json','w'),indent=2)



M1 vs M0
  dRecall @1% : +0.016667  CI95 [+0.000000, +0.033472]
  dFPR matched: -0.004531  CI95 [-0.017101, -0.000086]

M2 vs M0
  dRecall @1% : +0.011111  CI95 [-0.011111, +0.033333]
  dFPR matched: -0.003625  CI95 [-0.017461, +0.000463]

M3 vs M0
  dRecall @1% : -0.016667  CI95 [-0.033472, +0.000000]
  dFPR matched: +0.008350  CI95 [-0.000431, +0.033245]


In [11]:

# ============================================================
# 9. Run-level criterion outcomes + M3 negative-control flag
# ============================================================
def criterion_status(comp):
    b=comp['fixed_budget']['recall']
    a=comp['matched_recall']
    A=bool(a['point']<0 and a['ci_hi']<0)
    B=bool(b['point']>0 and b['ci_lo']>0)
    return A,B

A_M2,B_M2=criterion_status(paired['M2'])
A_M3,B_M3=criterion_status(paired['M3'])
criteria_met=bool(A_M2 or B_M2)
negative_control_flag=bool(A_M3 or B_M3)

run_level_outcome='CRITERIA MET' if criteria_met else 'CRITERIA NOT MET'
if SOURCE_MODE=='DEV':
    overall_status='NOT EVALUATED — development seed; no final H1-B claim permitted'
else:
    overall_status='PENDING — per-run final result only; apply frozen final-world rule separately'

verdict={
 'source_run_mode':SOURCE_MODE,
 'source_condition':SOURCE_CONDITION,
 'source_seed':SOURCE_SEED,
 'criterion_A_matched_recall_fpr':{
     'satisfied':A_M2,**paired['M2']['matched_recall']
 },
 'criterion_B_fixed_budget_recall':{
     'satisfied':B_M2,**paired['M2']['fixed_budget']['recall']
 },
 'run_level_outcome':run_level_outcome,
 'm3_negative_control_criterion_A':A_M3,
 'm3_negative_control_criterion_B':B_M3,
 'negative_control_flag':negative_control_flag,
 'overall_h1_b_status':overall_status,
}
json.dump(verdict,open(OUT/'run_level_verdict_retail.json','w'),indent=2)
print(json.dumps(verdict,indent=2))


{
  "source_run_mode": "FINAL",
  "source_condition": "B1_structured",
  "source_seed": 2105,
  "criterion_A_matched_recall_fpr": {
    "satisfied": false,
    "point": -0.0036246014085476366,
    "ci_lo": -0.017460703126338885,
    "ci_hi": 0.0004628334001485022,
    "fpr_ref": 0.009405611250028676,
    "fpr_new": 0.0057810098414810395,
    "achieved_recall_ref": 0.25,
    "achieved_recall_new": 0.25,
    "alerts_ref": 865,
    "alerts_new": 549,
    "tp_ref": 45,
    "tp_new": 45,
    "fp_ref": 820,
    "fp_new": 504,
    "n_boot_requested": 300,
    "n_boot_used": 300
  },
  "criterion_B_fixed_budget_recall": {
    "satisfied": false,
    "point": 0.011111111111111127,
    "ci_lo": -0.0111111111111111,
    "ci_hi": 0.033333333333333326,
    "n_boot": 1000
  },
  "run_level_outcome": "CRITERIA NOT MET",
  "m3_negative_control_criterion_A": false,
  "m3_negative_control_criterion_B": false,
  "negative_control_flag": false,
  "overall_h1_b_status": "PENDING \u2014 per-run final result

## 10. Describe detection by scenario

Load scenario identities after fitting and primary evaluation to show where the detections occur. These tables are descriptive and do not independently determine the run-level criterion outcome.

The remaining code cells create the per-run figures, check the evaluation records and export the result card, models, scores, hashes and ZIP package.


In [12]:

# ============================================================
# 10. Scenario recall diagnostics — post-hoc/descriptive
# ============================================================
scenario_path=ROOT/'qa'/'scenario_labels_PRIVATE.parquet'
scenario_rows=[]
if scenario_path.exists():
    scen=pd.read_parquet(scenario_path)
    scen['user']=scen.user.astype(str); scen['day']=pd.to_datetime(scen.day).dt.normalize()
    stest=base.loc[te,KEY+['label']].reset_index(drop=True).merge(scen[KEY+['scenario']],on=KEY,how='left')
    for cfg in ('M0','M1','M2','M3'):
        stest[f'{cfg}_alert']=ALERT[cfg]
    for scenario,g in stest.loc[stest.label.eq(1)].groupby('scenario',dropna=False):
        rec={'scenario':str(scenario),'malicious_days':int(len(g))}
        for cfg in ('M0','M1','M2','M3'):
            rec[f'{cfg}_detected_days']=int(g[f'{cfg}_alert'].sum())
            rec[f'{cfg}_recall']=float(g[f'{cfg}_alert'].mean()) if len(g) else np.nan
        scenario_rows.append(rec)
    scenario_df=pd.DataFrame(scenario_rows)
    scenario_df.to_csv(OUT/'scenario_detection_diagnostics_retail.csv',index=False)
    display(scenario_df)
else:
    print('scenario_labels_PRIVATE.parquet not present; skipping scenario diagnostics.')


,scenario,malicious_days,M0_detected_days,M0_recall,M1_detected_days,M1_recall,M2_detected_days,M2_recall,M3_detected_days,M3_recall
0,cross_role_misuse,108,9,0.083333,12,0.111111,11,0.101852,6,0.055556
1,globally_abnormal,36,36,1.000000,36,1.000000,36,1.000000,36,1.000000
2,low_and_slow,36,0,0.000000,0,0.000000,0,0.000000,0,0.000000


In [13]:

# ============================================================
# 11. Presentation-ready figures for this run
# ============================================================
# Figure 1: detected malicious days at the primary budget
fig,ax=plt.subplots(figsize=(8,4.5))
plot=primary.set_index('config').loc[['M0','M1','M2','M3']]
ax.bar(plot.index,plot.tp)
ax.set_ylabel('Malicious user-days detected')
ax.set_title(f'{SOURCE_CONDITION} — detections at the 1% alert budget ({SOURCE_MODE}, seed {SOURCE_SEED})')
for i,v in enumerate(plot.tp): ax.text(i,float(v)+0.2,str(int(v)),ha='center')
fig.tight_layout(); fig.savefig(FIGS/'primary_detections_retail.png',dpi=180); plt.close(fig)

# Figure 2: M2-M0 paired differences and 95% intervals
b=paired['M2']['fixed_budget']['recall']; a=paired['M2']['matched_recall']
vals=[a['point'],b['point']]
lo=[a['point']-a['ci_lo'],b['point']-b['ci_lo']]
hi=[a['ci_hi']-a['point'],b['ci_hi']-b['point']]
fig,ax=plt.subplots(figsize=(8,4.5))
x=np.arange(2)
ax.errorbar(x,vals,yerr=[lo,hi],fmt='o',capsize=5)
ax.axhline(0,linewidth=1)
ax.set_xticks(x,['ΔFPR matched recall\n(negative favours M2)','ΔRecall at 1%\n(positive favours M2)'])
ax.set_title(f'M2 − M0 paired differences ({SOURCE_MODE}, seed {SOURCE_SEED})')
fig.tight_layout(); fig.savefig(FIGS/'m2_vs_m0_intervals_retail.png',dpi=180); plt.close(fig)

# Figure 3: recall budget curves
fig,ax=plt.subplots(figsize=(8,4.5))
for cfg in ('M0','M2','M3'):
    g=curves[curves.config.eq(cfg)]
    ax.plot(g.budget_pct,g.recall,marker='o',label=cfg)
ax.set_xlabel('Alert budget (%)'); ax.set_ylabel('Recall'); ax.set_title('Recall across descriptive alert budgets')
ax.legend(); fig.tight_layout(); fig.savefig(FIGS/'recall_budget_curves_retail.png',dpi=180); plt.close(fig)

print('figures saved:',len(list(FIGS.glob('*.png'))))


figures saved: 3


In [14]:

# ============================================================
# 12. Final QA, result card, hashes, ZIP
# ============================================================
checks=[]
def q(name,cond,detail=''):
    cond=bool(cond); checks.append({'check':name,'passed':cond,'detail':str(detail)})
    print(('PASS' if cond else 'FAIL')+':',name,('— '+str(detail) if detail else ''))
    return cond

q('7R-B input hashes all verified',verify_df.passed.all(),f'{int(verify_df.passed.sum())}/{len(verify_df)}')
q('source feature-preparation stage had no model verdict',summary.get('h1_b_verdict') is None)
q('benign-only train preserved',int(y[tr].sum())==0)
q('test has exactly 180 malicious days for v1.0',int(yt.sum())==180,int(yt.sum()))
q('test has exactly 60 malicious accounts for v1.0',len(set(ut[yt.astype(bool)]))==60,len(set(ut[yt.astype(bool)])))
q('all four score arrays finite',all(np.isfinite(S[c]).all() for c in S))
q('equal alert counts at 1% budget',len({int(ALERT[c].sum()) for c in ALERT})==1,{c:int(ALERT[c].sum()) for c in ALERT})
q('M0 dimension = 41',len(FEATURES['M0'])==41)
q('M2 and M3 dimensions = 59',len(FEATURES['M2'])==59 and len(FEATURES['M3'])==59)
q('M1 dimension follows frozen 7R-B contract',len(FEATURES['M1'])==41+int(contract['m1_context_feature_count']),len(FEATURES['M1']))
q('run-level outcome derived from M2 only',run_level_outcome in ('CRITERIA MET','CRITERIA NOT MET'))
q('development run cannot emit final H1-B support claim',not (SOURCE_MODE=='DEV' and overall_status.startswith('H1-B SUPPORTED')),overall_status)

qa=pd.DataFrame(checks); qa.to_csv(QA/'7rc_structural_checks.csv',index=False)
assert qa.passed.all(),'7R-C structural QA failed.'

m0=primary[primary.config.eq('M0')].iloc[0]
m2=primary[primary.config.eq('M2')].iloc[0]
result_card={
 'environment':'B — Controlled Retail-and-Distribution Simulation',
 'condition':SOURCE_CONDITION,'run_mode':SOURCE_MODE,'generator_seed':SOURCE_SEED,
 'scientific_status':'development diagnostic' if SOURCE_MODE=='DEV' else 'reserved-final per-run result',
 'test_user_days':int(te.sum()),'test_positive_user_days':int(yt.sum()),
 'test_malicious_accounts':int(len(set(ut[yt.astype(bool)]))),
 'alerts_per_configuration':int(ALERT['M0'].sum()),
 'M0':{'tp':int(m0.tp),'fp':int(m0.fp),'recall':float(m0.recall),'precision':float(m0.precision),'fpr':float(m0.fpr),'average_precision':float(m0.average_precision)},
 'M2':{'tp':int(m2.tp),'fp':int(m2.fp),'recall':float(m2.recall),'precision':float(m2.precision),'fpr':float(m2.fpr),'average_precision':float(m2.average_precision)},
 'criterion_A':verdict['criterion_A_matched_recall_fpr'],
 'criterion_B':verdict['criterion_B_fixed_budget_recall'],
 'run_level_outcome':run_level_outcome,
 'negative_control_flag':negative_control_flag,
 'overall_h1_b_status':overall_status,
 'environment_A_status':'not read or modified by this notebook; locked CERT r6.2 result remains separate',
}
json.dump(result_card,open(OUT/'environment_b_run_result_card.json','w'),indent=2)

run_manifest={
 'notebook':'7R_C_retail_development_evaluation','stage':'7R-C',
 'source':{'run_mode':SOURCE_MODE,'condition':SOURCE_CONDITION,'seed':SOURCE_SEED},
 'detector':{'name':'IsolationForest','n_estimators':IF_ESTIMATORS,'max_samples':IF_MAX_SAMPLES,
             'contamination':'auto','scaler':'RobustScaler','seed':SEED,
             'selection':'transferred from Environment A; no retail detector-family selection'},
 'evaluation':{'primary_budget_pct':K_PRIMARY,'budget_grid':K_GRID,
               'paired_budget_bootstraps':N_BOOT_BUDGET,'matched_recall_bootstraps':N_BOOT_MATCHED,
               'ci_percent':95},
 'feature_counts':{cfg:len(FEATURES[cfg]) for cfg in FEATURES},
 'versions':{'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__,'scikit_learn':sklearn.__version__},
 'runtime_minutes':round((time.time()-T0)/60,3),
 'final_h1_b_claim_permitted':False,
}
json.dump(run_manifest,open(OUT/'run_manifest_retail.json','w'),indent=2)

hash_rows=[]
for p in sorted(OUT.rglob('*')):
    if p.is_file() and p.name not in ('retail_7rc_output_sha256.csv',):
        hash_rows.append({'relative_path':str(p.relative_to(OUT)),'size_bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(hash_rows).to_csv(OUT/'retail_7rc_output_sha256.csv',index=False)

zip_path=WORK/f'{OUT.name}_results.zip'
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix('')),'zip',root_dir=OUT)

print('\n'+'='*78)
print('7R-C COMPLETE')
print('='*78)
print(json.dumps(result_card,indent=2))
print(f"\nQA: {int(qa.passed.sum())}/{len(qa)} passed")
print('ZIP:',zip_path,f'({zip_path.stat().st_size/1e6:.1f} MB)')
print(f"Runtime: {(time.time()-T0)/60:.1f} min")
if SOURCE_MODE=='DEV':
    print('\nDEVELOPMENT DIAGNOSTIC ONLY — DO NOT PRESENT THIS AS FINAL H1-B SUPPORT.')
print('\nUpload the results ZIP for review before any reserved-final seed is run.')


PASS: 7R-B input hashes all verified — 14/14
PASS: source feature-preparation stage had no model verdict 
PASS: benign-only train preserved 
PASS: test has exactly 180 malicious days for v1.0 — 180
PASS: test has exactly 60 malicious accounts for v1.0 — 60
PASS: all four score arrays finite 
PASS: equal alert counts at 1% budget — {'M0': 873, 'M1': 873, 'M2': 873, 'M3': 873}
PASS: M0 dimension = 41 
PASS: M2 and M3 dimensions = 59 
PASS: M1 dimension follows frozen 7R-B contract — 61
PASS: run-level outcome derived from M2 only 
PASS: development run cannot emit final H1-B support claim — PENDING — per-run final result only; apply frozen final-world rule separately

7R-C COMPLETE
{
  "environment": "B \u2014 Controlled Retail-and-Distribution Simulation",
  "condition": "B1_structured",
  "run_mode": "FINAL",
  "generator_seed": 2105,
  "scientific_status": "reserved-final per-run result",
  "test_user_days": 87362,
  "test_positive_user_days": 180,
  "test_malicious_accounts": 60,
  "

## Read and retain the per-run package

Check the source mode, condition and seed in the result card before interpreting any table. In this supplied execution they identify final seed 2105.

A development-world result is an engineering diagnostic. A reserved-final result contributes evidence under the documented supplementary rule. Neither the current seed's outcome nor a pooled total replaces the primary seed-2101 decision.

Retain the package and its hash records. The five-world reporting stage is 7R-D. Editing these notes does not rerun the world, change the saved per-run outcome or independently establish the protocol-freeze chronology.
